# CancerVerso · V1
**Notebook de preparación de datos para el dashboard de Streamlit**

Qué hace, en orden:
1. Instala librerías y monta Google Drive (`MyDrive/Big data Zucco/` con `raw`, `procesados`, `modelos`, `stats`).
2. Descarga los metadatos y las máscaras de tumor de todo el dataset (0.55 GB).
3. Reparte los pacientes entre las cuentas de Colab (`PARTE` y `N_PARTES` de la sección 2) y mide las 13 máscaras de cada caso de **esta parte** **sin bajar ningún CT**.
4. Para hígado: baja CTs de esta parte, segmenta el órgano con TotalSegmentator, calcula **HTB%** y guarda volúmenes reducidos.
5. Control de calidad visual y curvas de seguimiento.
6. (Opcional, lo corre Isay al final.) Junta los CSV de todas las partes y recalcula las cifras globales. Ver sección 13.

**Antes de empezar**
- Entorno de ejecución → Cambiar tipo de entorno → **GPU T4** y **RAM alta (High-RAM)**. En una máquina de 8 GB de RAM, TotalSegmentator falló 8 veces por memoria (código -9).
- Opcional: guarda un token de Hugging Face en los secretos de Colab (icono de llave) con el nombre `HF_TOKEN`. Da más límite de descarga. **Cada cuenta debe usar su propio token** y, si Hugging Face lo pide, aceptar los términos del dataset BodyMaps/CancerVerse.
- Si van a repartir el trabajo entre amigos: cada uno abre **su propia copia** del notebook (Archivo → Guardar una copia en Drive), monta **su propio Drive** y cambia **solo** `PARTE` en la sección 2. Todo el equipo usa el mismo `N_PARTES`. No trabajen sobre el mismo archivo de Colab.

**Reanudable:** si Colab se desconecta, vuelve a correr todo desde arriba. Lo ya hecho se salta solo.

**Estructura real del dataset**
```
CancerVerse/CV_XXXXXXXX/ct.nii.gz                               <- tomografía (Hugging Face)
CancerVerse/CV_XXXXXXXX/segmentations/<órgano>_lesion.nii.gz    <- 13 máscaras de TUMOR (archivo aparte)
```
El dataset **no trae máscara del hígado como órgano**, solo `liver_lesion` (el tumor). Por eso el hígado se segmenta con TotalSegmentator.


## 1. Instalación y Drive
Se fija `TotalSegmentator==2.18.0` para que todo el equipo obtenga los mismos resultados.

**Ojo:** después del `pip`, Colab puede mostrar el botón **"Reiniciar sesión"**. Si aparece, reinicia y vuelve a correr desde esta celda (la segunda vez ya no lo pide).

In [ ]:
# ============================================================
# 1. INSTALAR LIBRERÍAS Y MONTAR GOOGLE DRIVE
# ============================================================
# TotalSegmentator se fija en la versión 2.18.0 para que todos obtengamos el mismo resultado.
!pip -q install TotalSegmentator==2.18.0 nibabel scikit-image huggingface_hub

from google.colab import drive
drive.mount("/content/drive")

## 2. Configuración
Rutas, parámetros, filtros y umbrales, todos como constantes en un solo lugar.

**Drive es la bodega y el disco local es la mesa de trabajo:** cada lote se procesa en `/content` y solo el resultado se guarda en Drive.

**Reparto entre cuentas:** en la celda de abajo está `N_PARTES` (en cuántas cuentas se reparte el trabajo; puede ser cualquier entero ≥ 1; 3 o 4 es lo típico) y `PARTE` (qué tajada le toca a esta cuenta: 0, 1, 2, ...). Cada amigo cambia **solo** `PARTE`. Todo el equipo debe usar el **mismo** `N_PARTES`. Los archivos de salida llevan el sufijo `_parte0`, `_parte1`, etc., para no pisarse al juntarlos.

**Umbrales clínicos (provisionales):**
- **Semáforo de HTB% (25% y 50%):** criterio provisional del equipo. Todavía no tiene fuente clínica.
- **Tendencia del volumen (+73% / −66%):** equivalente en volumen de RECIST 1.1 (Eisenhauer et al., *Eur J Cancer* 2009), que define progresión como +20% y respuesta como −30% del **diámetro**. Un +20% de diámetro equivale a cerca de +73% de volumen (1.2³ = 1.73). Es una aproximación, porque RECIST suma diámetros de lesiones elegidas y aquí se usa el volumen total.


In [ ]:
# ============================================================
# 2. CONFIGURACIÓN: rutas, carpetas y constantes del proyecto
# ============================================================
import os
import io
import gzip
import json
import time
import shutil
import tarfile
import hashlib
import datetime
import subprocess

import numpy as np
import pandas as pd
import nibabel as nib

# --- Rutas ---
BASE = "/content/drive/MyDrive/Big data Zucco"     # carpeta del proyecto en Drive (lo que debe sobrevivir a la sesión)
TMP = "/content/zucco_tmp"                # disco local de Colab: rápido, pero se borra al desconectar

# Si Drive no está montado, las carpetas se crearían en el disco local y se perderían. Mejor detenerse.
if not os.path.exists("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive no está montado: corre primero la sección 1.")

DIR_RAW = BASE + "/raw"                   # CTs y máscaras originales
DIR_PROC = BASE + "/procesados"           # volúmenes reducidos y cortes para la app
DIR_MODELOS = BASE + "/modelos"           # pesos de la U-Net (para la siguiente versión)
DIR_STATS = BASE + "/stats"               # tablas CSV de ESTA parte (con sufijo _parteN)
DIR_PARTES = BASE + "/stats_partes"       # aquí junta Isay los CSV que le entregan los amigos
DIR_UNION = BASE + "/stats_union"         # resultado de juntar todas las partes (nombres sin sufijo, los que lee la app)

for carpeta in [DIR_RAW, DIR_PROC, DIR_MODELOS, DIR_STATS, DIR_PARTES, DIR_UNION, TMP, DIR_PROC + "/higado"]:
    os.makedirs(carpeta, exist_ok=True)

# --- Origen de los datos ---
REPO_HF = "BodyMaps/CancerVerse"
URL_ETIQUETAS = "https://www.cs.jhu.edu/~zongwei/dataset/CancerVerse_Label.tar.gz"
RUTA_TAR = DIR_RAW + "/CancerVerse_Label.tar.gz"
RUTA_META = DIR_RAW + "/CancerVerse_dataset_metadata.csv"

# --- Los 13 órganos: nombre del archivo de máscara -> nombre en español ---
ORGANOS = {
    "liver": "Hígado", "pancreatic": "Páncreas", "kidney": "Riñón", "stomach": "Estómago",
    "adrenal_gland": "Glándula suprarrenal", "colon": "Colon", "bladder": "Vejiga",
    "prostate": "Próstata", "esophagus": "Esófago", "spleen": "Bazo", "uterus": "Útero",
    "duodenum": "Duodeno", "gallbladder": "Vesícula biliar",
}

# --- Partición entre cuentas de Colab ---
# N_PARTES: en cuántas cuentas se reparte el trabajo. Puede ser cualquier entero >= 1.
#           Lo típico es 3 o 4. TODO el equipo debe usar el MISMO número.
# PARTE: qué tajada le toca a ESTA cuenta (0, 1, 2, ... hasta N_PARTES - 1).
#        Cada amigo cambia SOLO esta línea. El resto del notebook no se toca.
# Ejemplo con 4 cuentas: un amigo pone PARTE = 0, otro 1, otro 2 y el último 3.
N_PARTES = 3
PARTE = 0

if type(N_PARTES) is not int or N_PARTES < 1:
    raise ValueError("N_PARTES debe ser un entero mayor o igual a 1.")
if type(PARTE) is not int or PARTE < 0 or PARTE >= N_PARTES:
    raise ValueError("PARTE debe ser un entero entre 0 y " + str(N_PARTES - 1) + " (ahora mismo N_PARTES = " + str(N_PARTES) + ").")

SUFIJO_PARTE = "_parte" + str(PARTE)

def parte_del_paciente(paciente):
    """Parte que le toca a un paciente. Es fija: el mismo paciente siempre cae en la misma parte."""
    # OJO: no uses hash() de Python. Ese valor cambia en cada sesión y mezclaría a los pacientes entre cuentas.
    return int(hashlib.md5(str(paciente).encode()).hexdigest(), 16) % N_PARTES

def agregar_columnas_parte(tabla):
    """Agrega al final las columnas parte y n_partes, para saber de qué tajada salió cada fila."""
    tabla = tabla.copy()
    tabla["parte"] = PARTE
    tabla["n_partes"] = N_PARTES
    return tabla

# --- Parámetros de esta versión ---
N_CASOS_HIGADO = 20            # CTs de hígado a procesar en esta versión (para llegar a 300 GB este número tiene que subir)
TAM_LOTE = 5                   # casos por lote
GUARDAR_CT_EN_DRIVE = True     # True = conservar el CT original en Drive
SEMILLA = 42

# --- Filtros de calidad ---
VOL_MIN_CM3 = 1.0              # tumores más chicos que esto son poco confiables para medir
SALTO_REVISAR = 10             # cambio de volumen (veces) entre estudios seguidos que se marca para revisar
DZ_MAX_MM = 5.0                # grosor de corte máximo de una serie "normal"
NZ_MIN = 50                    # número mínimo de cortes
DX_MAX_MM = 3.0                # tamaño máximo de píxel
QC_PCT_FUERA = 10              # % de tumor fuera del hígado a partir del cual se genera imagen de QC

# --- Umbrales clínicos (los mismos valores están en app.py) ---
# Semáforo de HTB%: CRITERIO PROVISIONAL DEL EQUIPO. No tiene fuente clínica todavía.
HTB_MODERADA = 25
HTB_ALTA = 50
# Tendencia del volumen: equivalente en volumen de RECIST 1.1 (Eisenhauer et al., Eur J Cancer 2009).
# RECIST habla de DIÁMETRO: progresión = +20%, respuesta = -30%. Si el tumor fuera una esfera,
# +20% de diámetro son 1.2^3 = 1.73 veces el volumen (+73%), y -30% son 0.7^3 = 0.34 veces (-66%).
# Es una aproximación: RECIST suma diámetros de lesiones elegidas, no volumen total. CRITERIO PROVISIONAL.
CAMBIO_CRECE_PCT = 73
CAMBIO_REDUCE_PCT = -66

# --- Token de Hugging Face (opcional) ---
# Con token las descargas tienen más límite. Se guarda en Colab: icono de llave -> nuevo secreto "HF_TOKEN".
# Cada cuenta de Colab debe crear y guardar el suyo.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    print("Sin HF_TOKEN en los secretos de Colab: se descarga sin token (funciona, con menos límite).")

print("Carpeta del proyecto:", BASE)
print("Esta cuenta trabaja la parte", PARTE, "de", N_PARTES, "| sufijo de archivos:", SUFIJO_PARTE)


## 3. Metadatos y máscaras
Baja el CSV de metadatos y el archivo con las máscaras de tumor. Se guardan en `raw/` y no se vuelven a descargar.

Limpieza:
- Fechas como fecha y edad como número (`"above 89"` se convierte en 90).
- **Sexo y edad faltantes** se rellenan con los otros estudios del mismo paciente: el sexo con el primero conocido y la edad con el promedio de sus otros estudios.

Después se reparte el dataset por paciente (con un hash fijo, no el `hash()` de Python) y esta cuenta se queda solo con su `PARTE`. Eso ocurre **antes** de cualquier límite o muestreo. Todos los CTs de un mismo paciente quedan en la misma parte, para no romper el seguimiento. El metadata de esta parte se guarda en `stats/metadata_parteN.csv`.


In [ ]:
# ============================================================
# 3. DESCARGAR METADATOS Y MÁSCARAS (una sola vez, quedan en Drive)
# ============================================================
import urllib.request
from huggingface_hub import hf_hub_download

# 3a. CSV de metadatos (24.5 MB): una fila por CT
if not os.path.exists(RUTA_META):
    ruta = hf_hub_download(REPO_HF, "CancerVerse_dataset_metadata.csv", repo_type="dataset", local_dir=TMP, token=HF_TOKEN)
    shutil.copy(ruta, RUTA_META)

# 3b. Máscaras de tumor de todo el dataset (0.55 GB comprimido)
if not os.path.exists(RUTA_TAR):
    print("Descargando máscaras (0.55 GB)...")
    urllib.request.urlretrieve(URL_ETIQUETAS, TMP + "/etiquetas.tmp")
    shutil.move(TMP + "/etiquetas.tmp", RUTA_TAR)

# Copia en el disco local: leer el tar desde Drive es mucho más lento
RUTA_TAR_LOCAL = TMP + "/CancerVerse_Label.tar.gz"
if not os.path.exists(RUTA_TAR_LOCAL):
    shutil.copy(RUTA_TAR, RUTA_TAR_LOCAL)

# 3c. Limpieza de metadatos
meta = pd.read_csv(RUTA_META)
meta = meta.rename(columns={
    "CancerVerse ID": "caso", "Patient ID": "paciente", "Encrypted Accession Number": "estudio",
    "sex": "sexo", "age": "edad", "phase": "fase", "exam_date": "fecha", "icd10_code": "icd10", "report": "reporte"})
meta = meta[["caso", "paciente", "estudio", "sexo", "edad", "fase", "fecha", "icd10", "reporte"]]

meta["fecha"] = pd.to_datetime(meta["fecha"], format="%m-%d-%Y", errors="coerce")
meta["edad"] = meta["edad"].replace("above 89", "90")          # el dataset escribe "above 89" para mayores de 89
meta["edad"] = pd.to_numeric(meta["edad"], errors="coerce")
meta["fase"] = meta["fase"].fillna("sin_dato")

# 3d. Rellenar sexo y edad faltantes con los otros estudios del MISMO paciente
sexo_antes = meta["sexo"].isna().sum()
edad_antes = meta["edad"].isna().sum()
sexo_del_paciente = meta.groupby("paciente")["sexo"].transform("first")              # primer sexo conocido del paciente
edad_del_paciente = meta.groupby("paciente")["edad"].transform("mean").round()       # edad promedio de sus otros estudios
meta["sexo"] = meta["sexo"].fillna(sexo_del_paciente)
meta["edad"] = meta["edad"].fillna(edad_del_paciente)
print("Sexo faltante:", sexo_antes, "->", meta["sexo"].isna().sum())
print("Edad faltante:", edad_antes, "->", meta["edad"].isna().sum())
print("Todo el dataset:", len(meta), "CTs |", meta["paciente"].nunique(), "pacientes |", meta["estudio"].nunique(), "estudios")
print("CTs sin fecha (todo el dataset):", meta["fecha"].isna().sum())

# 3e. Repartir los pacientes entre las cuentas (ANTES de cualquier límite o muestreo)
# Todos los CTs de un mismo paciente van a la misma parte, para no romper el seguimiento.
meta["parte"] = meta["paciente"].map(parte_del_paciente)
meta["n_partes"] = N_PARTES

print("Reparto en", N_PARTES, "partes:")
for p in range(N_PARTES):
    m = meta[meta["parte"] == p]
    if p == PARTE:
        marca = "  <- esta cuenta"
    else:
        marca = ""
    print("  Parte", p, ":", len(m), "CTs |", m["paciente"].nunique(), "pacientes" + marca)

meta = meta[meta["parte"] == PARTE].copy()
RUTA_META_PARTE = DIR_STATS + "/metadata" + SUFIJO_PARTE + ".csv"
meta.to_csv(RUTA_META_PARTE, index=False)

print(len(meta), "CTs en el metadata de esta parte |", meta["paciente"].nunique(), "pacientes |", meta["estudio"].nunique(), "estudios")
print("CTs sin fecha (esta parte):", int(meta["fecha"].isna().sum()))


## 4. Medir las máscaras
Recorre el archivo de máscaras una sola vez, caso por caso, y cuenta los vóxeles de tumor de cada órgano. Con el tamaño del vóxel se convierte a **cm³**. Solo se miden los casos de **esta parte**.

**El truco de velocidad:** dentro de un mismo caso, todas las máscaras vacías pesan exactamente lo mismo, porque son puros ceros del mismo tamaño. Solo se descomprimen las que pesan distinto (cerca del 4%). Sin esto habría que descomprimir 317,460 archivos.

**Validación del truco:** de las máscaras que se saltan, se descomprime 1 de cada 1,500 (unas 200 en todo el dataset; en una parte serán menos) para confirmar que de verdad están vacías. Si alguna tuviera tumor, la celda se detiene con un error.

Va en un solo proceso, sin paralelismo, para que sea fácil de seguir. La primera vez tarda bastante; después lee el resultado guardado (`volumen_por_caso_parteN.csv` y `casos_con_mascara_parteN.csv`).

**`LIMITE_CASOS`:** sirve solo para una prueba rápida y se aplica **después** de filtrar por parte. Una prueba no se guarda en Drive.


In [ ]:
# ============================================================
# 4. MEDIR LAS MÁSCARAS (sin bajar ningún CT, solo los de esta parte)
# ============================================================
REVISAR_CADA = 1500        # de las máscaras que se saltan por "vacías", se comprueba 1 de cada 1,500
LIMITE_CASOS = None        # para una prueba rápida pon un número (ej. 300); None = todos los casos de ESTA parte
casos_de_esta_parte = set(meta["caso"])

def leer_cabecera(datos):
    """Lee solo la cabecera de una máscara .nii.gz: forma (nx, ny, nz) y tamaño de vóxel en mm."""
    with gzip.open(io.BytesIO(datos)) as f:
        cabecera = nib.Nifti1Header.from_fileobj(f)
    return cabecera.get_data_shape(), cabecera.get_zooms()

def contar_voxeles(datos):
    """Descomprime una máscara completa y cuenta cuántos vóxeles son distintos de cero."""
    imagen = nib.Nifti1Image.from_bytes(gzip.decompress(datos))
    return int(np.count_nonzero(np.asanyarray(imagen.dataobj)))

def medir_caso(caso, archivos, filas_tumor, filas_caso, control):
    """Mide las 13 máscaras de un caso. 'archivos' es un diccionario {órgano: bytes del .nii.gz}."""
    # Todas las máscaras VACÍAS de un caso pesan lo mismo (son puros ceros del mismo tamaño).
    # El tamaño que más se repite es el de "vacía".
    tamanos = [len(datos) for datos in archivos.values()]
    tam_vacia = pd.Series(tamanos).mode()[0]
    patron_claro = tamanos.count(tam_vacia) >= 7      # si menos de 7 pesan igual, no nos fiamos del truco

    forma, vox = leer_cabecera(next(iter(archivos.values())))
    filas_caso.append([caso, forma[0], forma[1], forma[2], float(vox[0]), float(vox[1]), float(vox[2])])
    cm3_por_voxel = float(vox[0]) * float(vox[1]) * float(vox[2]) / 1000

    for organo, datos in archivos.items():
        parece_vacia = patron_claro and len(datos) == tam_vacia
        if parece_vacia:
            control["saltadas"] += 1
            if control["saltadas"] % REVISAR_CADA != 0:
                continue                              # se salta sin descomprimir
            control["revisadas"] += 1                 # esta sí se comprueba, para validar el truco
        n = contar_voxeles(datos)
        if n > 0:
            if parece_vacia:
                control["saltadas_con_tumor"] += 1    # sería un fallo del truco
            filas_tumor.append([caso, organo, n, n * cm3_por_voxel])

def medir_mascaras(ruta_tar):
    """Recorre el tar una sola vez. Solo mide los casos de esta parte. LIMITE_CASOS se aplica después del filtro."""
    filas_tumor, filas_caso = [], []
    control = {"saltadas": 0, "revisadas": 0, "saltadas_con_tumor": 0}
    caso_actual, archivos = None, {}
    t0 = time.time()
    with tarfile.open(ruta_tar, "r|gz") as tar:
        for miembro in tar:
            if not miembro.name.endswith("_lesion.nii.gz"):
                continue
            partes = miembro.name.split("/")               # .../CV_00000001/segmentations/liver_lesion.nii.gz
            caso = partes[-3]
            organo = partes[-1].replace("_lesion.nii.gz", "")
            if caso != caso_actual and archivos:           # cambió el caso: se mide el anterior
                medir_caso(caso_actual, archivos, filas_tumor, filas_caso, control)
                archivos = {}
                if len(filas_caso) % 1000 == 0:
                    print(len(filas_caso), "casos |", len(filas_tumor), "tumores |", round(time.time() - t0), "s")
                if LIMITE_CASOS is not None and len(filas_caso) >= LIMITE_CASOS:
                    break
            caso_actual = caso
            if caso in casos_de_esta_parte:
                archivos[organo] = tar.extractfile(miembro).read()
    if archivos:                                           # el último caso del archivo (si era de esta parte)
        medir_caso(caso_actual, archivos, filas_tumor, filas_caso, control)

    tumores = pd.DataFrame(filas_tumor, columns=["caso", "organo", "voxeles", "vol_cm3"])
    casos = pd.DataFrame(filas_caso, columns=["caso", "nx", "ny", "nz", "dx", "dy", "dz"])
    return tumores, casos, control

RUTA_TUMORES = DIR_STATS + "/volumen_por_caso" + SUFIJO_PARTE + ".csv"
RUTA_CASOS = DIR_STATS + "/casos_con_mascara" + SUFIJO_PARTE + ".csv"

def csv_de_esta_particion(ruta):
    """True si el CSV existe y se hizo con el mismo PARTE y N_PARTES de ahora."""
    if not os.path.exists(ruta):
        return False
    tabla = pd.read_csv(ruta)
    if "parte" not in tabla.columns or "n_partes" not in tabla.columns or len(tabla) == 0:
        return False
    return int(tabla["parte"].iloc[0]) == PARTE and int(tabla["n_partes"].iloc[0]) == N_PARTES

if csv_de_esta_particion(RUTA_TUMORES) and csv_de_esta_particion(RUTA_CASOS):
    print("Ya estaba calculado, se reutiliza. (Para repetirlo, borra esos dos CSV de la carpeta stats.)")
    tumores = pd.read_csv(RUTA_TUMORES)
    casos = pd.read_csv(RUTA_CASOS)
else:
    if os.path.exists(RUTA_TUMORES) or os.path.exists(RUTA_CASOS):
        print("Había CSV de otra partición (otro PARTE o N_PARTES). Se vuelven a medir las máscaras de esta parte.")
    print("Midiendo máscaras de la parte", PARTE, ":", len(casos_de_esta_parte), "CTs de esta cuenta")
    tumores, casos, control = medir_mascaras(RUTA_TAR_LOCAL)
    tumores = agregar_columnas_parte(tumores)
    casos = agregar_columnas_parte(casos)
    # Validación del truco: ninguna de las máscaras saltadas que se revisaron debe tener tumor
    print("Máscaras saltadas por vacías:", control["saltadas"])
    print("Revisadas como muestra:", control["revisadas"], "| con tumor:", control["saltadas_con_tumor"])
    assert control["saltadas_con_tumor"] == 0, "El truco de 'las vacías pesan igual' falló: hay que medir todas."
    if LIMITE_CASOS is None:
        tumores.to_csv(RUTA_TUMORES, index=False)
        casos.to_csv(RUTA_CASOS, index=False)
    else:
        # Una prueba parcial NO se guarda: si se guardara, la próxima vez se reutilizaría como si fuera la corrida completa
        print("Prueba con", LIMITE_CASOS, "casos: no se guarda en Drive.")
print(len(casos), "casos con máscara |", len(tumores), "anotaciones de tumor (CT x órgano)")


## 5. Base única, estadísticas por órgano y seguimiento

**Una sola base para todas las cifras.** El metadata tiene 24,422 CTs, pero las máscaras son de 24,420: `CV_00011576` y `CV_00018563` no traen máscaras. Todas las cifras de esta parte se calculan sobre los CTs **con máscaras que le tocan a esta cuenta**. Las cifras de todo el dataset se recalculan al final, en la sección 13, cuando Isay junta las partes.

**Tabla por órgano.** La edad media y el % de mujeres se calculan **por paciente** (una fila por paciente), no por CT.

**Seguimiento de hígado (`seguimiento_higado_parteN.csv`)**
- Solo entran **fechas con tumor hepático anotado**. Un estudio sin tumor anotado no aparece en la curva, así que la curva no muestra cuándo un tumor desapareció.
- Se excluyen los CTs sin fecha, porque no se pueden ordenar en el tiempo.
- **Misma fase:** para cada paciente se prefiere la fase portal venosa en todas las fechas. Si en una fecha no hay CT en esa fase, se usa el que exista y la columna `misma_fase` queda en `False`.
- Si en una fecha quedan varios CTs, se toma **la fila** de mayor volumen (`idxmax`), para que caso, fase y volumen sean del mismo CT.
- **`revisar = True`** cuando el volumen cambia más de 10 veces respecto al estudio anterior, o cuando es menor a 1 cm³. Son mediciones poco confiables: conviene verlas antes de sacar conclusiones.

Este cálculo va **por paciente**, así que no necesita ver a los pacientes de las otras partes. Por eso es seguro repartir antes.


In [ ]:
# ============================================================
# 5. BASE ÚNICA, TABLA POR ÓRGANO Y SEGUIMIENTO DE HÍGADO
# ============================================================
def tabla_por_organo(t, casos):
    """Estadísticas por órgano. Edad y sexo se calculan por paciente, no por CT."""
    filas = []
    for clave, nombre in ORGANOS.items():
        o = t[t["organo"] == clave]

        # Edad y sexo se calculan POR PACIENTE (una fila por paciente: su primer estudio con tumor),
        # para que un paciente con muchos CTs no cuente varias veces.
        pacientes = o.sort_values("fecha").drop_duplicates("paciente")
        con_sexo = pacientes["sexo"].notna().sum()
        mujeres = (pacientes["sexo"] == "F").sum()

        # Pacientes con tumor de este órgano en 2 o más FECHAS distintas
        con_fecha = o[o["fecha"].notna()]
        fechas_por_paciente = con_fecha.groupby("paciente")["fecha"].nunique()

        filas.append({
            "organo": clave,
            "nombre": nombre,
            "n_casos": len(casos),
            "n_con_tumor": len(o),
            "pct_con_tumor": round(100 * len(o) / len(casos), 2),
            "vol_tumoral_medio_cm3": round(o["vol_cm3"].mean(), 2),
            "vol_tumoral_mediana_cm3": round(o["vol_cm3"].median(), 2),
            "n_pacientes_con_tumor": o["paciente"].nunique(),
            "pacientes_2mas_fechas": int((fechas_por_paciente >= 2).sum()),
            "edad_media": round(pacientes["edad"].mean(), 1),
            "pct_mujeres": round(100 * mujeres / con_sexo, 1),
        })
    return pd.DataFrame(filas).sort_values("n_con_tumor", ascending=False)

def armar_seguimiento_higado(t):
    """Seguimiento de hígado a partir de las anotaciones (sin CTs). Una fila por paciente y fecha."""
    # Se reinicia el índice para que idxmax no se confunda si la tabla viene de juntar varias partes.
    h = t[(t["organo"] == "liver") & (t["fecha"].notna())].copy().reset_index(drop=True)
    h["fecha"] = pd.to_datetime(h["fecha"], errors="coerce")
    h = h[h["fecha"].notna()].copy()

    # Fase preferida de cada paciente: portal venosa si cubre tantas fechas como la que más;
    # si no, la fase con más fechas. Comparar siempre la misma fase evita saltos falsos de volumen.
    fase_preferida = {}
    for paciente, g in h.groupby("paciente"):
        fechas_por_fase = g.groupby("fase")["fecha"].nunique()
        if "portal_venous" in fechas_por_fase.index and fechas_por_fase["portal_venous"] == fechas_por_fase.max():
            fase_preferida[paciente] = "portal_venous"
        else:
            fase_preferida[paciente] = fechas_por_fase.idxmax()
    h["es_fase_preferida"] = h["fase"] == h["paciente"].map(fase_preferida)

    # En cada fecha: si hay CT en la fase preferida, se usan solo esos; si no hay, se usa lo que exista
    hay_preferida = h.groupby(["paciente", "fecha"])["es_fase_preferida"].transform("max")
    h = h[h["es_fase_preferida"] == hay_preferida]

    # De los CTs que quedan en cada fecha se toma LA FILA de mayor volumen (idxmax devuelve su índice),
    # así el caso, la fase y el volumen son siempre del mismo CT.
    filas_max = h.groupby(["paciente", "fecha"])["vol_cm3"].idxmax()
    seg = h.loc[filas_max, ["paciente", "fecha", "caso", "fase", "vol_cm3"]]

    # Solo pacientes con 2 o más fechas
    n_fechas = seg.groupby("paciente")["fecha"].transform("count")
    seg = seg[n_fechas >= 2].sort_values(["paciente", "fecha"]).copy()
    seg["dias"] = (seg["fecha"] - seg.groupby("paciente")["fecha"].transform("min")).dt.days
    seg["misma_fase"] = seg.groupby("paciente")["fase"].transform("nunique") == 1

    # Marcas de revisión: salto de más de 10x respecto al estudio anterior, o volumen menor a 1 cm³
    vol_anterior = seg.groupby("paciente")["vol_cm3"].shift(1)
    razon = seg["vol_cm3"] / vol_anterior
    seg["salto_10x"] = (razon > SALTO_REVISAR) | (razon < 1 / SALTO_REVISAR)
    seg["revisar"] = seg["salto_10x"] | (seg["vol_cm3"] < VOL_MIN_CM3)
    return seg.reset_index(drop=True)

def armar_resumen(meta_todas, meta_con_mascara, t, sin_mascara, n_sin_fecha):
    """Cifras globales. meta_todas es el metadata (de esta parte o de la unión) antes de quitar los CTs sin máscara."""
    fechas = meta_con_mascara[meta_con_mascara["fecha"].notna()].groupby("paciente")["fecha"]
    dias_seguimiento = (fechas.max() - fechas.min()).dt.days
    if len(dias_seguimiento) == 0:
        max_anios = 0
    else:
        max_anios = round(float(dias_seguimiento.max()) / 365.25, 1)
    return {
        "n_cts": int(len(meta_con_mascara)),
        "n_cts_dataset": int(len(meta_todas)),
        "casos_sin_mascara": list(sin_mascara),
        "n_cts_sin_fecha": int(n_sin_fecha),
        "n_pacientes": int(meta_con_mascara["paciente"].nunique()),
        "n_estudios": int(meta_con_mascara["estudio"].nunique()),
        "n_organos": len(ORGANOS),
        "n_anotaciones_tumor": int(len(t)),
        "n_cts_con_tumor": int(t["caso"].nunique()),
        "seguimiento_max_anios": max_anios,
    }

# --- 5a. Una sola base: los CTs de esta parte que SÍ tienen máscaras ---
sin_mascara = sorted(set(meta["caso"]) - set(casos["caso"]))
print("CTs del metadata de esta parte sin máscaras (se excluyen):", sin_mascara)
n_sin_fecha = int(meta["fecha"].isna().sum())
meta_todas = meta.copy()                   # se guarda antes de quitar los que no tienen máscara (sirve para el resumen)
meta = meta[meta["caso"].isin(casos["caso"])].copy()
print("Base de trabajo de esta parte:", len(meta), "CTs |", meta["paciente"].nunique(), "pacientes |", meta["estudio"].nunique(), "estudios")

# Cada fila de 't' es una anotación de tumor (un CT x un órgano) con los datos de su CT
# Se quitan parte y n_partes de tumores para que no se dupliquen al unir con meta (que ya las trae).
t = tumores.drop(columns=["parte", "n_partes"], errors="ignore").merge(meta, on="caso", how="inner")

# --- 5b. Tabla por órgano ---
stats_organos = agregar_columnas_parte(tabla_por_organo(t, casos))
stats_organos.to_csv(DIR_STATS + "/stats_organos" + SUFIJO_PARTE + ".csv", index=False)

# --- 5c. Seguimiento de hígado (sale solo de las máscaras, sin CTs) ---
seg = agregar_columnas_parte(armar_seguimiento_higado(t))
seg.to_csv(DIR_STATS + "/seguimiento_higado" + SUFIJO_PARTE + ".csv", index=False)

pacientes_revisar = seg.groupby("paciente")["revisar"].max().sum()
print("Seguimiento de hígado (esta parte):", seg["paciente"].nunique(), "pacientes,", len(seg), "estudios")
print("  Saltos de más de 10x:", int(seg["salto_10x"].sum()), "| estudios con menos de 1 cm³:", int((seg["vol_cm3"] < VOL_MIN_CM3).sum()))
print("  Pacientes con algún estudio marcado para revisar:", int(pacientes_revisar))
print("  Pacientes con la misma fase en todas sus fechas:", int(seg.groupby("paciente")["misma_fase"].first().sum()) if len(seg) else 0)

# --- 5d. Cifras de esta parte para la portada (todas sobre la misma base) ---
resumen = armar_resumen(meta_todas, meta, t, sin_mascara, n_sin_fecha)
resumen["parte"] = PARTE
resumen["n_partes"] = N_PARTES
with open(DIR_STATS + "/resumen_dataset" + SUFIJO_PARTE + ".json", "w") as f:
    json.dump(resumen, f, indent=2)

stats_organos


## 6. Selección de los 20 CTs de hígado
Se elige **dentro de esta parte** (los pacientes de las otras cuentas no entran):
- **Pacientes con seguimiento limpio (hasta 16 CTs):** 4 pacientes con 3 fechas y 2 con 2 fechas. Deben tener todos los tumores de 1 cm³ o más, ningún estudio marcado para revisar, la misma fase en todas las fechas y series normales. Si en esta parte no hay tantos, se usan los que haya.
- **Tumores grandes (el resto, hasta llegar a 20):** del 10% más grande, de pacientes sin seguimiento de **esta parte**. Sirven para probar el control de calidad. El umbral del 10% se calcula solo con los casos de esta parte.
- **Series normales:** cortes de 5 mm o menos, 50 cortes o más y píxel de 3 mm o menos.

La semilla fija hace que, para la misma parte, siempre salgan los mismos casos.


In [ ]:
# ============================================================
# 6. ELEGIR LOS 20 CTs DE HÍGADO (solo de esta parte)
# ============================================================
# Series "normales": cortes de 5 mm o menos, al menos 50 cortes y píxel de 3 mm o menos
serie_ok = casos[(casos["dz"] <= DZ_MAX_MM) & (casos["nz"] >= NZ_MIN) & (casos["dx"] <= DX_MAX_MM)]["caso"]
print("Series normales (esta parte):", len(serie_ok), "de", len(casos))

rng = np.random.default_rng(SEMILLA)

def elegir_al_azar(candidatos, cuantos):
    """Elige 'cuantos' elementos de la lista, siempre los mismos gracias a la semilla."""
    lista = sorted(candidatos)
    if cuantos <= 0:
        return []
    if len(lista) < cuantos:
        print("AVISO: se pedían", cuantos, "candidatos, pero en esta parte solo hay", len(lista), ". Se usan todos.")
        return lista
    return list(rng.choice(lista, size=cuantos, replace=False))

# --- 6a. Pacientes con seguimiento limpio: 4 con 3 fechas y 2 con 2 fechas (hasta 16 CTs) ---
seg["serie_ok"] = seg["caso"].isin(serie_ok)
por_paciente = seg.groupby("paciente").agg(
    n_fechas=("caso", "count"),
    vol_min=("vol_cm3", "min"),
    alguno_revisar=("revisar", "max"),
    todas_series_ok=("serie_ok", "min"),
    misma_fase=("misma_fase", "first"),
)
limpios = por_paciente[
    (por_paciente["vol_min"] >= VOL_MIN_CM3)            # ningún tumor menor a 1 cm³
    & (por_paciente["alguno_revisar"] == False)         # sin saltos de más de 10x
    & (por_paciente["todas_series_ok"] == True)         # sin series raras
    & (por_paciente["misma_fase"] == True)              # misma fase en todas las fechas
]
pacientes_3 = elegir_al_azar(limpios[limpios["n_fechas"] == 3].index, 4)
pacientes_2 = elegir_al_azar(limpios[limpios["n_fechas"] == 2].index, 2)
longitudinales = seg[seg["paciente"].isin(list(pacientes_3) + list(pacientes_2))]

# --- 6b. CTs sueltos con tumores grandes (para el control de calidad) ---
sueltos = t[(t["organo"] == "liver") & (t["fecha"].notna())]
sueltos = sueltos[~sueltos["paciente"].isin(seg["paciente"])]          # pacientes sin seguimiento
sueltos = sueltos[sueltos["caso"].isin(serie_ok)]
faltan = N_CASOS_HIGADO - len(longitudinales)
if len(sueltos) == 0 or faltan <= 0:
    casos_grandes = []
else:
    umbral_grande = sueltos["vol_cm3"].quantile(0.90)                  # el 10% de tumores más grandes de ESTA parte
    grandes = sueltos[sueltos["vol_cm3"] >= umbral_grande].drop_duplicates("paciente")
    casos_grandes = elegir_al_azar(grandes["caso"], faltan)

# --- 6c. Tabla final ---
elegidos = list(longitudinales["caso"]) + list(casos_grandes)
seleccion = t[(t["organo"] == "liver") & (t["caso"].isin(elegidos))]
seleccion = seleccion[["caso", "paciente", "fecha", "sexo", "edad", "fase", "icd10", "vol_cm3", "reporte"]]
seleccion = agregar_columnas_parte(seleccion)
seleccion = seleccion.sort_values(["paciente", "fecha"]).reset_index(drop=True)
seleccion.to_csv(DIR_STATS + "/seleccion_higado" + SUFIJO_PARTE + ".csv", index=False)

if len(seleccion) == 0:
    print("AVISO: en esta parte no hubo CTs de hígado que pasaran los filtros.")
else:
    print(len(seleccion), "CTs de", seleccion["paciente"].nunique(), "pacientes | volumen mínimo:", round(seleccion["vol_cm3"].min(), 2), "cm³")
seleccion.drop(columns="reporte")


## 7. Archivos de los casos elegidos
Una función saca del archivo de máscaras solo las de los casos elegidos. La otra baja el CT de Hugging Face cuando toca procesarlo.

In [ ]:
# ============================================================
# 7. TRAER LOS ARCHIVOS DE LOS CASOS ELEGIDOS
# ============================================================
def extraer_mascaras(lista_casos):
    """Saca del tar solo las máscaras de los casos pedidos y las deja en raw/CancerVerse/<caso>/segmentations/."""
    faltan = []
    for caso in lista_casos:
        if not os.path.exists(DIR_RAW + "/CancerVerse/" + caso + "/segmentations/liver_lesion.nii.gz"):
            faltan.append(caso)
    if len(faltan) == 0:
        return
    pendientes = len(faltan) * len(ORGANOS)
    with tarfile.open(RUTA_TAR_LOCAL, "r|gz") as tar:
        for miembro in tar:
            partes = miembro.name.split("/")
            if not miembro.isfile() or len(partes) < 3 or partes[-3] not in faltan:
                continue
            destino = DIR_RAW + "/CancerVerse/" + partes[-3] + "/segmentations"
            os.makedirs(destino, exist_ok=True)
            with open(destino + "/" + partes[-1], "wb") as f:
                f.write(tar.extractfile(miembro).read())
            pendientes = pendientes - 1
            if pendientes == 0:
                break                                   # ya están todas: no hace falta leer el resto

def descargar_ct(caso):
    """Baja un CT de Hugging Face al disco local y devuelve su ruta. Si ya está en Drive, usa ese."""
    en_drive = DIR_RAW + "/CancerVerse/" + caso + "/ct.nii.gz"
    if os.path.exists(en_drive):
        return en_drive
    return hf_hub_download(REPO_HF, "CancerVerse/" + caso + "/ct.nii.gz", repo_type="dataset", local_dir=TMP, token=HF_TOKEN)

extraer_mascaras(seleccion["caso"].tolist())
print("Máscaras listas para", len(seleccion), "casos")

## 8. Procesamiento de un CT

**Cómo se calcula el HTB%**

> **HTB% = vóxeles de tumor ÷ vóxeles de (hígado de TotalSegmentator ∪ tumor anotado) × 100**

El denominador es la **unión** de las dos máscaras, no la de TotalSegmentator sola. Motivo: TotalSegmentator a veces deja fuera del hígado los tumores grandes u oscuros. Si se dividiera solo entre su máscara, el hígado saldría más chico de lo que es y el porcentaje se inflaría (incluso arriba de 100). Con la unión, el tumor siempre cuenta como parte del hígado.

**Comprobaciones antes de calcular:** el CT, la máscara de tumor y la de hígado deben tener el mismo tamaño **y** la misma posición en el espacio (`np.allclose` de sus affines con tolerancia 0.001).

**TotalSegmentator** se llama con `subprocess.run` como un comando normal, solo para el hígado (`--roi_subset liver`). Con GPU va a resolución completa; sin GPU usa `--fast` y el resultado se marca como provisional (`ts_rapido = True`).

In [ ]:
# ============================================================
# 8. FUNCIONES PARA PROCESAR UN CT DE HÍGADO
# ============================================================
import torch
from scipy import ndimage
from PIL import Image

HAY_GPU = torch.cuda.is_available()
MODO_RAPIDO = not HAY_GPU      # con GPU: resolución completa. Sin GPU: modelo rápido de 3 mm (--fast)
print("GPU disponible:", HAY_GPU, "| modo rápido (--fast):", MODO_RAPIDO)
if MODO_RAPIDO:
    print("AVISO: sin GPU los resultados son provisionales. Activa la T4 y vuelve a correr desde la sección 1.")

VENTANA_HU = (-160, 240)       # ventana de tejido blando, la que se usa para ver hígado
LADO_PX = 256                  # ancho/alto del volumen reducido para la app
MAX_CORTES = 96                # número máximo de cortes del volumen reducido
MARGEN_MM = 15                 # margen arriba y abajo del hígado al recortar

def segmentar_higado(ruta_ct, carpeta_salida):
    """Corre TotalSegmentator solo para el hígado y devuelve la ruta de la máscara (liver.nii.gz)."""
    if HAY_GPU:
        dispositivo = "gpu"
    else:
        dispositivo = "cpu"
    orden = ["TotalSegmentator", "-i", ruta_ct, "-o", carpeta_salida, "--roi_subset", "liver", "--device", dispositivo, "--quiet"]
    if MODO_RAPIDO:
        orden.append("--fast")
    resultado = subprocess.run(orden, capture_output=True, text=True)
    ruta_mascara = carpeta_salida + "/liver.nii.gz"
    if resultado.returncode != 0 or not os.path.exists(ruta_mascara):
        # código -9 = el sistema cerró el proceso por falta de memoria RAM
        raise RuntimeError("TotalSegmentator falló (código " + str(resultado.returncode) + "): " + (resultado.stderr + resultado.stdout)[-300:])
    return ruta_mascara

def cargar_ras(ruta):
    """Carga un .nii.gz y lo reorienta a la convención RAS (x = derecha, y = anterior, z = superior)."""
    return nib.as_closest_canonical(nib.load(ruta))

def a_vista_axial(vol_xyz):
    """Reordena a [corte, fila, columna] como en radiología: anterior arriba y derecha del paciente a la izquierda."""
    volteado = vol_xyz[::-1, ::-1, :]
    return np.ascontiguousarray(np.transpose(volteado, (2, 1, 0)))

def componer_corte(ct, tumor, higado):
    """Pinta un corte en grises con el tumor en rojo y el contorno del hígado en cian."""
    rgb = np.stack([ct, ct, ct], axis=2).astype(np.float32)
    contorno = higado & ~ndimage.binary_erosion(higado)
    rgb[contorno] = (0, 229, 255)
    rgb[tumor] = 0.55 * rgb[tumor] + 0.45 * np.array([255, 59, 92])
    return rgb.astype(np.uint8)

def procesar_caso(caso, ruta_ct):
    """Segmenta el hígado, calcula HTB% y guarda el volumen reducido y un corte de muestra."""
    salida = DIR_PROC + "/higado/" + caso
    os.makedirs(salida, exist_ok=True)
    ruta_higado = segmentar_higado(ruta_ct, salida)

    img_ct = cargar_ras(ruta_ct)
    img_tumor = cargar_ras(DIR_RAW + "/CancerVerse/" + caso + "/segmentations/liver_lesion.nii.gz")
    img_higado = cargar_ras(ruta_higado)

    # Las tres imágenes deben tener el mismo tamaño Y estar en la misma posición del espacio (affine)
    for nombre, img in [("tumor", img_tumor), ("hígado", img_higado)]:
        if img.shape != img_ct.shape:
            raise ValueError("La máscara de " + nombre + " tiene otro tamaño que el CT")
        if not np.allclose(img_ct.affine, img.affine, atol=1e-3):
            raise ValueError("La máscara de " + nombre + " no está alineada con el CT (affine distinta)")

    tumor = np.asanyarray(img_tumor.dataobj) > 0
    higado_ts = np.asanyarray(img_higado.dataobj) > 0
    higado_total = higado_ts | tumor                   # hígado de TotalSegmentator UNIDO con el tumor anotado

    dx, dy, dz = [float(v) for v in img_ct.header.get_zooms()[:3]]
    cm3_por_voxel = dx * dy * dz / 1000

    # --- Métricas ---
    vox_tumor = int(tumor.sum())
    vox_higado = int(higado_total.sum())
    vox_fuera = int((tumor & ~higado_ts).sum())        # tumor que quedó fuera de la máscara de TotalSegmentator
    metricas = {
        "caso": caso,
        "vol_tumor_cm3": round(vox_tumor * cm3_por_voxel, 2),
        "vol_higado_cm3": round(vox_higado * cm3_por_voxel, 1),
        "htb_pct": round(100 * vox_tumor / vox_higado, 2),
        "n_lesiones": int(ndimage.label(tumor)[1]),    # zonas de tumor separadas entre sí
        "voxeles_tumor_fuera_higado": vox_fuera,
        "pct_tumor_fuera_higado": round(100 * vox_fuera / vox_tumor, 2),
        "ts_rapido": MODO_RAPIDO,                      # True = resultado provisional (--fast)
    }

    # --- Volumen reducido para la app ---
    # 1) Recortar en z a la zona del hígado, con un margen
    cortes_con_higado = np.where(higado_total.any(axis=(0, 1)))[0]
    margen = int(round(MARGEN_MM / dz))
    z0 = max(cortes_con_higado[0] - margen, 0)
    z1 = min(cortes_con_higado[-1] + margen + 1, higado_total.shape[2])

    # 2) Pasar el CT de unidades Hounsfield a 0-255 con la ventana de tejido blando
    ct = np.asanyarray(img_ct.dataobj[:, :, z0:z1]).astype(np.float32)
    ct = np.clip(ct, VENTANA_HU[0], VENTANA_HU[1])
    ct = (ct - VENTANA_HU[0]) / (VENTANA_HU[1] - VENTANA_HU[0]) * 255
    ct = ct.astype(np.uint8)

    # 3) Reducir el tamaño: 256 px de lado y como máximo 96 cortes
    f_xy = LADO_PX / max(ct.shape[0], ct.shape[1])
    f_z = min(1.0, MAX_CORTES / ct.shape[2])
    factores = (f_xy, f_xy, f_z)

    def reducir_mascara(mascara):
        # Se interpola como número y se corta en 0.3 (no 0.5) para no perder las lesiones pequeñas
        recorte = mascara[:, :, z0:z1].astype(np.float32)
        return ndimage.zoom(recorte, factores, order=1) >= 0.3

    ct_r = a_vista_axial(ndimage.zoom(ct, factores, order=1))
    tumor_r = a_vista_axial(reducir_mascara(tumor))
    higado_r = a_vista_axial(reducir_mascara(higado_total))
    higado_ts_r = a_vista_axial(reducir_mascara(higado_ts))          # máscara original de TotalSegmentator, para el QC
    espaciado = np.array([dz / f_z, dy / f_xy, dx / f_xy], dtype=np.float32)     # mm por [corte, fila, columna]
    np.savez_compressed(salida + "/volumen.npz", ct=ct_r, tumor=tumor_r, higado=higado_r, higado_ts=higado_ts_r, espaciado_mm=espaciado)

    # --- Corte de muestra: el que tiene más tumor ---
    k = int(tumor_r.sum(axis=(1, 2)).argmax())
    Image.fromarray(componer_corte(ct_r[k], tumor_r[k], higado_r[k])).save(salida + "/corte.png")
    metricas["corte_max_tumor"] = k
    metricas["n_cortes"] = int(ct_r.shape[0])
    return metricas

## 9. Procesar por lotes
Recorre los casos pendientes de 5 en 5. Cada caso terminado se anota en `checklist_higado_parteN.csv` y sus resultados en `casos_higado_parteN.csv`. Un caso que falla queda registrado con su error, no detiene a los demás y se reintenta en la siguiente corrida.


In [ ]:
# ============================================================
# 9. PROCESAR POR LOTES CON CHECKLIST (reanudable, de esta parte)
# ============================================================
RUTA_CHECKLIST = DIR_STATS + "/checklist_higado" + SUFIJO_PARTE + ".csv"
RUTA_RESULTADOS = DIR_STATS + "/casos_higado" + SUFIJO_PARTE + ".csv"

def agregar_fila(ruta, fila):
    """Agrega una fila (diccionario) al final de un CSV. Si el archivo no existe, lo crea con encabezado."""
    tabla = pd.DataFrame([fila])
    tabla.to_csv(ruta, mode="a", index=False, header=not os.path.exists(ruta))

def leer_checklist():
    """Devuelve el ÚLTIMO estado de cada caso (un caso que falló y luego salió bien cuenta como 'ok')."""
    if not os.path.exists(RUTA_CHECKLIST):
        return pd.DataFrame(columns=["caso", "estado", "detalle", "gb_ct", "segundos", "procesado_el", "parte", "n_partes"])
    return pd.read_csv(RUTA_CHECKLIST).drop_duplicates("caso", keep="last")

checklist = leer_checklist()
hechos = set(checklist[checklist["estado"] == "ok"]["caso"])
pendientes = []
for caso in seleccion["caso"]:
    if caso not in hechos:
        pendientes.append(caso)             # los que nunca se hicieron y los que fallaron se (re)intentan
print(len(hechos), "casos ya procesados |", len(pendientes), "pendientes")

for inicio in range(0, len(pendientes), TAM_LOTE):
    lote = pendientes[inicio:inicio + TAM_LOTE]
    print("\nLote", inicio // TAM_LOTE + 1, ":", lote)
    for caso in lote:
        t0 = time.time()
        ahora = datetime.datetime.now().isoformat(timespec="seconds")
        try:
            ruta_ct = descargar_ct(caso)
            gb = os.path.getsize(ruta_ct) / 1e9
            metricas = procesar_caso(caso, ruta_ct)

            # Se agregan los datos del paciente a las métricas del caso
            info = seleccion[seleccion["caso"] == caso].iloc[0]
            for columna in ["paciente", "fecha", "sexo", "edad", "fase", "icd10", "reporte"]:
                metricas[columna] = info[columna]
            metricas["parte"] = PARTE
            metricas["n_partes"] = N_PARTES
            agregar_fila(RUTA_RESULTADOS, metricas)

            # El CT original se guarda en Drive (o se borra) y se libera el disco local
            en_drive = DIR_RAW + "/CancerVerse/" + caso + "/ct.nii.gz"
            if ruta_ct != en_drive:
                if GUARDAR_CT_EN_DRIVE:
                    shutil.copy(ruta_ct, en_drive)
                os.remove(ruta_ct)

            agregar_fila(RUTA_CHECKLIST, {"caso": caso, "estado": "ok", "detalle": "", "gb_ct": round(gb, 3),
                                          "segundos": round(time.time() - t0), "procesado_el": ahora,
                                          "parte": PARTE, "n_partes": N_PARTES})
            print(" ", caso, "| HTB", metricas["htb_pct"], "% | tumor", metricas["vol_tumor_cm3"], "cm³ |", round(time.time() - t0), "s")
        except Exception as error:
            # Un caso con error no detiene a los demás: queda anotado y se reintenta la próxima vez
            agregar_fila(RUTA_CHECKLIST, {"caso": caso, "estado": "error", "detalle": str(error)[:200], "gb_ct": 0,
                                          "segundos": round(time.time() - t0), "procesado_el": ahora,
                                          "parte": PARTE, "n_partes": N_PARTES})
            print(" ", caso, "| ERROR:", error)

checklist = leer_checklist()
n_ok = (checklist["estado"] == "ok").sum()
n_error = (checklist["estado"] == "error").sum()
print("\nListo:", n_ok, "casos ok,", n_error, "con error |", round(checklist["gb_ct"].sum(), 2), "GB de CT procesados")


## 10. Control de calidad
Primero lista todos los casos con algo de tumor fuera de la máscara de hígado. Después genera la imagen de **todos los casos con más de 10% del tumor fuera** (se guardan en `stats/qc/`).
- **Cian:** contorno del hígado según TotalSegmentator.
- **Rojo:** tumor dentro del hígado.
- **Amarillo:** tumor que quedó fuera de la máscara de hígado.

In [ ]:
# ============================================================
# 10. CONTROL DE CALIDAD: ¿la máscara de hígado cubre al tumor?
# ============================================================
if not os.path.exists(RUTA_RESULTADOS):
    raise RuntimeError("Todavía no hay casos procesados: revisa los errores de la sección 9.")

import matplotlib.pyplot as plt

# Solo los casos de la selección actual (el CSV puede traer casos de corridas anteriores)
res = pd.read_csv(RUTA_RESULTADOS).drop_duplicates("caso", keep="last")
res = res[res["caso"].isin(seleccion["caso"])]

# --- Aviso por cada caso con vóxeles de tumor fuera de la máscara de hígado ---
fuera = res[res["voxeles_tumor_fuera_higado"] > 0].sort_values("pct_tumor_fuera_higado", ascending=False)
print(len(fuera), "de", len(res), "casos tienen algo de tumor fuera de la máscara de hígado de TotalSegmentator:")
for _, fila in fuera.iterrows():
    if fila["pct_tumor_fuera_higado"] > QC_PCT_FUERA:
        nivel = "REVISAR"
    else:
        nivel = "leve"
    print(" ", fila["caso"], "|", fila["pct_tumor_fuera_higado"], "% del tumor fuera |", int(fila["voxeles_tumor_fuera_higado"]), "vóxeles |", nivel)

def mostrar_qc(caso, n_cortes=4):
    """Muestra 4 cortes: contorno del hígado de TotalSegmentator (cian), tumor dentro (rojo) y tumor fuera (amarillo)."""
    v = np.load(DIR_PROC + "/higado/" + caso + "/volumen.npz")
    ct, tumor, ts = v["ct"], v["tumor"], v["higado_ts"]
    fila = res[res["caso"] == caso].iloc[0]

    # Se eligen cortes repartidos a lo largo de la zona con tumor
    cortes_con_tumor = np.where(tumor.any(axis=(1, 2)))[0]
    posiciones = np.linspace(0, len(cortes_con_tumor) - 1, n_cortes).astype(int)
    elegidos = cortes_con_tumor[posiciones]

    fig, ejes = plt.subplots(1, n_cortes, figsize=(4 * n_cortes, 4.4), facecolor="black")
    for eje, k in zip(ejes, elegidos):
        capa = np.zeros((ct.shape[1], ct.shape[2], 4))            # imagen transparente para pintar encima
        capa[tumor[k] & ts[k]] = (1.0, 0.23, 0.36, 0.55)          # rojo: tumor dentro del hígado
        capa[tumor[k] & ~ts[k]] = (1.0, 0.92, 0.0, 0.85)          # amarillo: tumor fuera del hígado
        eje.imshow(ct[k], cmap="gray", vmin=0, vmax=255)
        eje.imshow(capa)
        if ts[k].any():
            eje.contour(ts[k], levels=[0.5], colors="#00e5ff", linewidths=1.2)
        eje.set_title("corte " + str(k), color="white", fontsize=10)
        eje.axis("off")
    titulo = caso + " | tumor " + str(fila["vol_tumor_cm3"]) + " cm³ | HTB " + str(fila["htb_pct"]) + "% | " + str(fila["pct_tumor_fuera_higado"]) + "% del tumor fuera del hígado"
    fig.suptitle(titulo, color="white", fontsize=12)
    plt.tight_layout()
    os.makedirs(DIR_STATS + "/qc", exist_ok=True)
    fig.savefig(DIR_STATS + "/qc/" + caso + ".png", dpi=110, facecolor="black")
    plt.show()

# --- Imagen de QC de TODOS los casos con más de 10% del tumor fuera del hígado ---
casos_qc = fuera[fuera["pct_tumor_fuera_higado"] > QC_PCT_FUERA]["caso"].tolist()
print("\nCasos con más de", QC_PCT_FUERA, "% del tumor fuera del hígado:", len(casos_qc))
for caso in casos_qc:
    mostrar_qc(caso)

## 11. Curvas de seguimiento
Para los pacientes con 2 o más estudios procesados, grafica el volumen tumoral y el HTB% contra los días desde el primer estudio. La tendencia usa los umbrales provisionales de la sección 2. Los pacientes se identifican con su **ID corto** (los primeros 6 caracteres), igual que en la app. La gráfica se guarda como `curvas_seguimiento_parteN.png`.


In [ ]:
# ============================================================
# 11. CURVAS DE SEGUIMIENTO DE LOS CASOS PROCESADOS
# ============================================================
if not os.path.exists(RUTA_RESULTADOS):
    raise RuntimeError("Todavía no hay casos procesados: revisa los errores de la sección 9.")

res = pd.read_csv(RUTA_RESULTADOS, parse_dates=["fecha"]).drop_duplicates("caso", keep="last")
res["dias"] = (res["fecha"] - res.groupby("paciente")["fecha"].transform("min")).dt.days
res.to_csv(RUTA_RESULTADOS, index=False)          # se guarda ya con la columna "dias"

res = res[res["caso"].isin(seleccion["caso"])]
n_estudios = res.groupby("paciente")["caso"].transform("count")
con_seguimiento = res[n_estudios >= 2].sort_values(["paciente", "fecha"])

fig, (eje_vol, eje_htb) = plt.subplots(1, 2, figsize=(13, 4.5))
for paciente, g in con_seguimiento.groupby("paciente"):
    etiqueta = "Paciente " + paciente[:6]         # ID corto: los primeros 6 caracteres, igual que en la app
    eje_vol.plot(g["dias"], g["vol_tumor_cm3"], marker="o", label=etiqueta)
    eje_htb.plot(g["dias"], g["htb_pct"], marker="o", label=etiqueta)
eje_vol.set_title("Volumen tumoral hepático")
eje_vol.set_xlabel("Días desde el primer estudio")
eje_vol.set_ylabel("cm³")
eje_htb.set_title("Carga tumoral hepática (HTB%)")
eje_htb.set_xlabel("Días desde el primer estudio")
eje_htb.set_ylabel("% del hígado")
eje_vol.grid(alpha=0.3)
eje_htb.grid(alpha=0.3)
eje_vol.legend(fontsize=8)
plt.tight_layout()
fig.savefig(DIR_STATS + "/curvas_seguimiento" + SUFIJO_PARTE + ".png", dpi=130)
plt.show()

# Tendencia: cambio de volumen entre el primer y el último estudio (umbrales de la sección 2)
filas = []
for paciente, g in con_seguimiento.groupby("paciente"):
    vol_inicial = g["vol_tumor_cm3"].iloc[0]
    vol_final = g["vol_tumor_cm3"].iloc[-1]
    cambio = 100 * (vol_final - vol_inicial) / vol_inicial
    if cambio >= CAMBIO_CRECE_PCT:
        tendencia = "crece"
    elif cambio <= CAMBIO_REDUCE_PCT:
        tendencia = "se reduce"
    else:
        tendencia = "estable"
    filas.append({"paciente": paciente[:6], "estudios": len(g), "dias": int(g["dias"].max()),
                  "vol_inicial_cm3": vol_inicial, "vol_final_cm3": vol_final, "cambio_pct": round(cambio, 1), "tendencia": tendencia})
pd.DataFrame(filas)


## 12. Resumen
Muestra los GB de CT procesados de **esta parte** frente a la meta de 300 GB del equipo, avisa si quedan casos en modo rápido y lista los archivos de esta parte.

**Para ver el dashboard de todo el equipo:** primero hay que correr la sección 13 (juntar partes). Luego se descargan de Drive las carpetas `Big data Zucco/stats_union` (renómbrala a `stats`) y, si se decide compartirlos, `Big data Zucco/procesados`. Se ponen dentro de `datos/` junto a `app.py`. En la Mac, activa siempre el entorno `BigDataZucco` y corre `streamlit run app.py`.


In [ ]:
# ============================================================
# 12. RESUMEN: qué quedó en Drive y cuánto llevamos procesado (esta parte)
# ============================================================
if not os.path.exists(RUTA_RESULTADOS):
    raise RuntimeError("Todavía no hay casos procesados: revisa los errores de la sección 9.")

checklist = leer_checklist()
ok = checklist[checklist["estado"] == "ok"]
gb = ok["gb_ct"].sum()
print("Parte", PARTE, "de", N_PARTES)
print("CTs procesados en esta parte:", len(ok), "|", round(gb, 2), "GB | meta del equipo: 300 GB")
print("  (Esta cifra es solo de esta cuenta. El total se ve en la sección 13, cuando se juntan las partes.)")

res = pd.read_csv(RUTA_RESULTADOS).drop_duplicates("caso", keep="last")
if res["ts_rapido"].any():
    print("AVISO: hay casos procesados con --fast. La app mostrará 'valores provisionales' hasta reprocesarlos con GPU.")
else:
    print("Todos los casos se procesaron a resolución completa (sin --fast).")

print("\nArchivos de esta parte (en stats/). Entrégaselos a Isay; NO compartas la carpeta raw/ ni los CTs:")
for carpeta in [DIR_STATS, DIR_PROC + "/higado"]:
    for raiz, subcarpetas, archivos in os.walk(carpeta):
        for nombre in sorted(archivos):
            if nombre.endswith((".csv", ".json", ".npz", ".png")):
                ruta = os.path.join(raiz, nombre)
                print(" ", os.path.relpath(ruta, BASE), "|", round(os.path.getsize(ruta) / 1e6, 2), "MB")


## 13. Juntar partes (opcional, lo corre Isay)

Cuando cada cuenta ya terminó, Isay junta **solo los CSV y los JSON** de todas las partes y recalcula las estadísticas globales y el seguimiento sobre la unión. **Las stats de cada parte no se promedian.**

**Qué se comparte y qué no**
- Sí: los archivos de `Big data Zucco/stats/` que terminan en `_parte0`, `_parte1`, etc. (CSV, JSON y, si quieres, la gráfica PNG).
- No: los CTs ni las máscaras de `raw/` (la licencia del dataset es **CC BY-NC-ND**). Cada cuenta se queda con sus CTs en su propio Drive.

**Cómo se usa**
1. Cada amigo le entrega a Isay sus archivos `_parteN`.
2. Isay los copia todos a `MyDrive/Big data Zucco/stats_partes/` (la carpeta se crea sola en la sección 2).
3. Si Colab se reinició, corre antes las secciones 1, 2 y 5 (la 5 define las funciones de recálculo; si los CSV de su parte ya existen, no vuelve a medir las máscaras).
4. Corre esta celda. El resultado queda en `MyDrive/Big data Zucco/stats_union/`, con los nombres que ya espera la app (`stats_organos.csv`, `seguimiento_higado.csv`, `resumen_dataset.json`, etc.).

La celda acepta cualquier `N_PARTES`: lee todos los `*_parte*.csv` que encuentre, comprueba que todos traigan el mismo `n_partes`, avisa si falta alguna parte y verifica que ningún paciente esté en dos partes.


In [ ]:
# ============================================================
# 13. JUNTAR PARTES (opcional; lo corre Isay cuando ya tiene los CSV)
# ============================================================
# Si Colab se reinició, corre primero las secciones 1, 2 y 5.
if "tabla_por_organo" not in dir() or "armar_seguimiento_higado" not in dir() or "armar_resumen" not in dir():
    raise RuntimeError("Primero corre las secciones 1, 2 y 5. La sección 5 define las funciones para recalcular las stats y el seguimiento.")

os.makedirs(DIR_PARTES, exist_ok=True)
os.makedirs(DIR_UNION, exist_ok=True)

def listar_archivos_parte(prefijo, extension=".csv"):
    """Busca prefijo_parteN.ext en stats_partes/ y devuelve un diccionario {N: ruta}."""
    encontrados = {}
    cola = prefijo + "_parte"
    for nombre in sorted(os.listdir(DIR_PARTES)):
        if not nombre.startswith(cola) or not nombre.endswith(extension):
            continue
        medio = nombre[len(cola):-len(extension)]
        if medio.isdigit():
            encontrados[int(medio)] = os.path.join(DIR_PARTES, nombre)
    return encontrados

def leer_partes(prefijo, extension=".csv", obligatorio=True):
    """Lee y concatena todos los archivos de un prefijo. Devuelve (tabla o None, {parte: ruta})."""
    archivos = listar_archivos_parte(prefijo, extension)
    if len(archivos) == 0:
        if obligatorio:
            raise RuntimeError("No encontré " + prefijo + "_parte*" + extension + " en " + DIR_PARTES + ". Copia ahí los archivos de cada amigo.")
        print("AVISO: no hay", prefijo + "_parte*" + extension, "(se omite).")
        return None, {}

    tablas = []
    for p in sorted(archivos):
        ruta = archivos[p]
        if extension == ".json":
            with open(ruta) as f:
                tabla = pd.json_normalize(json.load(f))
        elif prefijo in ("metadata", "seguimiento_higado", "seleccion_higado", "casos_higado"):
            tabla = pd.read_csv(ruta, parse_dates=["fecha"])
        else:
            tabla = pd.read_csv(ruta)
        if "parte" not in tabla.columns or "n_partes" not in tabla.columns:
            raise RuntimeError(ruta + " no trae las columnas parte y n_partes. ¿Es un archivo de esta versión?")
        valores_parte = set(tabla["parte"].dropna().unique())
        if valores_parte != {p}:
            raise RuntimeError(ruta + " tiene parte=" + str(sorted(valores_parte)) + " pero el nombre dice parte" + str(p) + ".")
        tablas.append(tabla)
        print(" ", ruta, ":", len(tabla), "filas")
    return pd.concat(tablas, ignore_index=True), archivos

print("Leyendo archivos de", DIR_PARTES)
meta_u, arch_meta = leer_partes("metadata")
tumores_u, arch_tum = leer_partes("volumen_por_caso")
casos_u, arch_casos = leer_partes("casos_con_mascara")

# Todos deben haberse generado con el mismo N_PARTES
n_vistos = set(meta_u["n_partes"].unique()) | set(tumores_u["n_partes"].unique()) | set(casos_u["n_partes"].unique())
if len(n_vistos) != 1:
    raise RuntimeError("Las partes no coinciden en n_partes: " + str(sorted(n_vistos)) + ". Alguien corrió el notebook con un N_PARTES distinto.")
n_partes_union = int(list(n_vistos)[0])
print("N_PARTES según los archivos:", n_partes_union)

def avisar_faltantes(nombre, archivos):
    encontradas = set(archivos)
    faltan = sorted(set(range(n_partes_union)) - encontradas)
    extra = sorted(encontradas - set(range(n_partes_union)))
    print(" ", nombre, "→ partes", sorted(encontradas))
    if faltan:
        print("    AVISO: faltan las partes", faltan, "de", nombre + ". Las cifras globales quedan incompletas.")
    if extra:
        print("    AVISO: hay partes inesperadas", extra, "en", nombre)
    return faltan

faltan = []
faltan += avisar_faltantes("metadata", arch_meta)
faltan += avisar_faltantes("volumen_por_caso", arch_tum)
faltan += avisar_faltantes("casos_con_mascara", arch_casos)
completo = len(faltan) == 0
if completo:
    print("Están todas las partes de 0 a", n_partes_union - 1, "(cobertura completa).")
else:
    print("AVISO: la unión está incompleta. Se calcula con lo que hay, pero no es el dataset completo.")

# Ningún paciente puede estar en dos partes
partes_por_paciente = meta_u.groupby("paciente")["parte"].nunique()
traslape = partes_por_paciente[partes_por_paciente > 1]
if len(traslape) > 0:
    ejemplos = ", ".join(traslape.index[:10].astype(str))
    raise RuntimeError("Hay " + str(len(traslape)) + " pacientes en más de una parte (no se debe juntar así). Ejemplos: " + ejemplos)
print("Traslape de Patient ID entre partes: 0 (bien)")
print("Unión:", len(meta_u), "CTs |", meta_u["paciente"].nunique(), "pacientes")

# Recalcular stats y seguimiento sobre la UNIÓN (no se promedian las stats de cada parte)
sin_mascara_u = sorted(set(meta_u["caso"]) - set(casos_u["caso"]))
n_sin_fecha_u = int(meta_u["fecha"].isna().sum())
meta_con_mascara_u = meta_u[meta_u["caso"].isin(casos_u["caso"])].copy()
t_u = tumores_u.drop(columns=["parte", "n_partes"], errors="ignore").merge(meta_con_mascara_u, on="caso", how="inner")

stats_u = tabla_por_organo(t_u, casos_u)
stats_u["n_partes"] = n_partes_union
seg_u = armar_seguimiento_higado(t_u)
# En el seguimiento unido se conserva la columna parte de cada paciente (viene de metadata)
parte_de = meta_con_mascara_u.drop_duplicates("paciente").set_index("paciente")["parte"]
seg_u["parte"] = seg_u["paciente"].map(parte_de)
seg_u["n_partes"] = n_partes_union

resumen_u = armar_resumen(meta_u, meta_con_mascara_u, t_u, sin_mascara_u, n_sin_fecha_u)
resumen_u["n_partes"] = n_partes_union
resumen_u["partes_incluidas"] = sorted(set(arch_meta) | set(arch_tum) | set(arch_casos))
resumen_u["union_completa"] = completo

meta_u.to_csv(DIR_UNION + "/metadata.csv", index=False)
tumores_u.to_csv(DIR_UNION + "/volumen_por_caso.csv", index=False)
casos_u.to_csv(DIR_UNION + "/casos_con_mascara.csv", index=False)
stats_u.to_csv(DIR_UNION + "/stats_organos.csv", index=False)
seg_u.to_csv(DIR_UNION + "/seguimiento_higado.csv", index=False)
with open(DIR_UNION + "/resumen_dataset.json", "w") as f:
    json.dump(resumen_u, f, indent=2)

print("\nCifras de la unión:")
print("  CTs con máscara:", resumen_u["n_cts"], "| CTs en el metadata:", resumen_u["n_cts_dataset"])
print("  Pacientes:", resumen_u["n_pacientes"], "| anotaciones de tumor:", resumen_u["n_anotaciones_tumor"])
print("  CTs sin máscara:", resumen_u["casos_sin_mascara"])
print("  Seguimiento de hígado:", seg_u["paciente"].nunique(), "pacientes,", len(seg_u), "estudios")
print("  Saltos de más de 10x:", int(seg_u["salto_10x"].sum()))

# Tablas opcionales (si algún amigo ya procesó CTs)
for prefijo in ["seleccion_higado", "casos_higado", "checklist_higado"]:
    tabla_o, arch_o = leer_partes(prefijo, obligatorio=False)
    if tabla_o is None:
        continue
    avisar_faltantes(prefijo, arch_o)
    if "paciente" in tabla_o.columns:
        chocan = tabla_o.groupby("paciente")["parte"].nunique()
        chocan = chocan[chocan > 1]
        if len(chocan) > 0:
            raise RuntimeError("En " + prefijo + " hay pacientes en más de una parte.")
    tabla_o.to_csv(DIR_UNION + "/" + prefijo + ".csv", index=False)
    print("  Unión de", prefijo + ":", len(tabla_o), "filas")

print("\nListo. Las tablas juntas están en:", DIR_UNION)
print("La app busca estos nombres (sin _parteN), así que para ver el dashboard usa esta carpeta como datos/stats/.")
print("Los volúmenes reducidos (procesados/higado/) se quedan en el Drive de cada cuenta: no se copian aquí.")
